In [80]:
# simple example to demo cyclic

In [81]:
import random
from typing import TypedDict, Literal
from enum import Enum

from langgraph.graph import StateGraph, START, END
from langgraph.types import Command

In [82]:
class State(TypedDict):
    random_number: int
    current_value: int
    iter_count: int
    message: str
    messages: list[str]


class NodeName(str, Enum):
    GENERATE_NUMBER = "generate_number"
    NODE2 = "node 2"
    NODE3 = "node 3"

MAX_CNT = 3

In [83]:
def generate_number(state: State) -> Command[Literal[NodeName.NODE2]]:
    """Node 1: Generate a random number and route conditionally"""
    print(f"\n## Now in iteration {state.get('iter_count')}")
    print("\n----> Now in node1")
    num = random.randint(1, 100)
    print(f"\tGenerated random number: {num}")
    
    # Create message for this node
    node_message = f"Node 1: Generated number {num}"
    
    # Append to existing messages
    messages = state.get("messages", [])
    messages.append(node_message)
    
    return Command(
        update={
            "random_number": num,
            "current_value": num, 
            "message": f"Generated number: {num}",
            "messages": messages,
            "iter_count": state.get("iter_count", 0) + 1,
        },
        goto=NodeName.NODE2
    )


def node_2(state: State) -> Command[Literal[NodeName.NODE3]]:
    
    print("----> Now in node2")
    
    # print(f"Message: {state['message']}")
    multiplier = 5
    
    current_value = state.get("current_value", 0)
    current_value*= multiplier

    node_message = f"Node 2: Multiplying {state.get('current_value', 0)} with {multiplier} gives {current_value}"
    print(f"\tCurrent value: {current_value}")

    # Append to existing messages
    messages = state.get("messages", [])
    messages.append(node_message)
    
    return Command(
        update={
            "current_value": current_value,
            "message": f"Current value is {current_value}",
            "messages": messages,
        },
        goto=NodeName.NODE3,
    )


def node_3(state: State) -> Command[Literal[END]]:
    """decision node"""

    print("----> Now in node3")
    add_val = 3

    current_value = state.get("current_value")
    current_value+= add_val
    print(f"\tCurrent value: {current_value}")

    # Create message for this node
    node_message = f"Node 3: Adding {add_val} to {state.get('current_value')} gives {current_value}"
        
    if current_value % 7 == 0:
        next_node = END
        node_message+= "\nBINGO!!!! We got it!"
    elif state.get("iter_count") > MAX_CNT: 
        next_node = END
        node_message+= "\nUnfortunately the max iter count has reached"
    else:
        next_node = NodeName.GENERATE_NUMBER
        node_message+= "\nGoing to fetch a random number again"

    # Append to existing messages
    messages = state.get("messages", [])
    messages.append(node_message)

    return Command(
        update={
            "message": f"Current value is {current_value}",
            "messages": messages
        },
        goto=next_node
    )

In [84]:
def create_graph() -> StateGraph:
    """Create the conditional routing graph"""
    graph = StateGraph(State)
    
    # Add nodes
    graph.add_node(NodeName.GENERATE_NUMBER, generate_number)
    graph.add_node(NodeName.NODE2, node_2)
    graph.add_node(NodeName.NODE3, node_3)
    
    # Add edges
    graph.add_edge(START, NodeName.GENERATE_NUMBER)
    
    return graph.compile()


In [85]:
def run_simple_graph():
    """Run the simple graph example"""
    graph = create_graph()
    
    # Initial state
    initial_state = {
        "random_number": 0, "message": "Starting...", "messages": [],
        "current_value": 0, "iter_count": 0
    }
    
    print(" Running Simple Conditional Graph")
    print("=" * 50)
    
    # Run the graph
    result = graph.invoke(initial_state)
    
    print("\n\n Final State:")
    print(f"a) Random Number: {result['random_number']}")
    print(f"b) Current value: {result['current_value']}")
    print(f"c) Iteration count: {result['iter_count']}")
    print(f"d) Message: {result['message']}")
    print("e) All Messages from Graph Execution:")
    print("\t", "-" * 40)
    for i, msg in enumerate(result['messages'], 1):
        print(f"\t{i}. {msg}")
    
    return result

In [86]:
if __name__ == "__main__":
    # Run multiple times to see different routing
    
    num_cnts = 1
    for i in range(num_cnts):
        print(f"\n Run {i+1}:")
        res = run_simple_graph()



 Run 1:
 Running Simple Conditional Graph

## Now in iteration 0

----> Now in node1
	Generated random number: 18
----> Now in node2
	Current value: 90
----> Now in node3
	Current value: 93

## Now in iteration 1

----> Now in node1
	Generated random number: 54
----> Now in node2
	Current value: 270
----> Now in node3
	Current value: 273


 Final State:
a) Random Number: 54
b) Current value: 270
c) Iteration count: 2
d) Message: Current value is 273
e) All Messages from Graph Execution:
	 ----------------------------------------
	1. Node 1: Generated number 18
	2. Node 2: Multiplying 18 with 5 gives 90
	3. Node 3: Adding 3 to 90 gives 93
Going to fetch a random number again
	4. Node 1: Generated number 54
	5. Node 2: Multiplying 54 with 5 gives 270
	6. Node 3: Adding 3 to 270 gives 273
BINGO!!!! We got it!
